# 3 · TinyGPT: from the Transformer to a language model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajdama/paper-to-pytorch/blob/main/notebooks/03_tinygpt.ipynb)

The bonus steps of [paper-to-pytorch](https://github.com/rajdama/paper-to-pytorch): drop the encoder, keep the
masked decoder, and train it to predict the next character of Shakespeare. Then build a
BPE tokenizer and a KV cache.

**Use a GPU:** *Runtime → Change runtime type → T4 GPU*. Training takes about 10 minutes.

In [ ]:
# Get the code (Colab already has PyTorch and matplotlib installed)
import os
if not os.path.exists("step03_attention"):
    if not os.path.exists("paper-to-pytorch"):
        !git clone -q https://github.com/rajdama/paper-to-pytorch.git
    %cd paper-to-pytorch

import torch
from IPython.display import Image, display
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU is fine for this notebook)")

## Train a bigger TinyGPT on the GPU
About 10M parameters, 256 characters of context. (The CI version on GitHub is much smaller
because it runs on a CPU.)

In [ ]:
!python -m bonus.step13_gpt_decoder_only.train_gpt \
    --steps 4000 --d-model 384 --layers 6 --heads 6 --block-size 256 \
    --batch-size 64 --dropout 0.2 --lr 1e-3 --eval-every 500 --sample-tokens 600
display(Image("bonus/step13_gpt_decoder_only/loss_char.png"))

## Play with sampling
- **temperature** < 1: safer and more repetitive; > 1: more creative and more broken
- **top-k / top-p**: cut off the unlikely tail of the distribution

In [ ]:
for temperature in (0.5, 1.0, 1.4):
    print(f"========== temperature {temperature} ==========")
    !python -m bonus.step13_gpt_decoder_only.generate --prompt "JULIET:" --tokens 250 --temperature {temperature}

In [ ]:
!python -m bonus.step13_gpt_decoder_only.generate --prompt "KING HENRY:" --tokens 250 --temperature 0.9 --top-p 0.9

## Bonus 14 · BPE tokenizer
Byte Pair Encoding merges the most frequent pairs of tokens over and over. Watch what it learns.

In [ ]:
!python -m bonus.step14_bpe_tokenizer.demo

Now train TinyGPT on BPE tokens instead of characters. Each token covers several characters, so the same context window sees much more text.

In [ ]:
!python -m bonus.step13_gpt_decoder_only.train_gpt --tokenizer bpe --vocab-size 1024 \
    --steps 4000 --d-model 384 --layers 6 --heads 6 --block-size 256 \
    --batch-size 64 --dropout 0.2 --eval-every 500 --sample-tokens 200

> Note: the char and BPE losses can't be compared directly; one is per character, the
> other per token. Compare the samples instead.

## Bonus 15 · KV cache
Without a cache, generating each new token re-processes the whole context. With one, each
step processes a single token.

In [ ]:
!python -m bonus.step15_kv_cache.benchmark
display(Image("bonus/step15_kv_cache/benchmark.png"))

In [ ]:
# Generate with the cache. Same seed without --kv-cache gives identical text, just slower.
%time !python -m bonus.step13_gpt_decoder_only.generate --prompt "ROMEO:" --tokens 240 --kv-cache
%time !python -m bonus.step13_gpt_decoder_only.generate --prompt "ROMEO:" --tokens 240